In [1]:
import pandas as pd
import warnings

# 忽略 pandas 的一些非致命警告，让输出更干净
warnings.filterwarnings('ignore')

# ================= 配置区 =================
RAW_KG_PATH = "kg.csv"  # 确保这个大文件还在你的目录下
SEED_KEYWORD = "Alzheimer"  # ★★★ 修改点：关键词改为阿尔茨海默

def inspect_data():
    print(f"🕵️ 正在侦察 PrimeKG ({RAW_KG_PATH}) 关于 '{SEED_KEYWORD}' 的数据 ...")
    
    try:
        # 读取 PrimeKG (文件很大，low_memory=False 防止 DtypeWarning)
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    except FileNotFoundError:
        print(f"❌ 错误：找不到 {RAW_KG_PATH}，请检查路径。")
        return

    # 1. 在原始数据中找 "Alzheimer"
    print(f"   PrimeKG 原始总行数: {len(df)}")
    
    # 模糊匹配任何包含 Alzheimer 的行 (不论大小写)
    mask = (df['x_name'].str.contains(SEED_KEYWORD, case=False, na=False)) | \
           (df['y_name'].str.contains(SEED_KEYWORD, case=False, na=False))
    
    ad_df = df[mask]
    print(f"👉 包含 '{SEED_KEYWORD}' 的相关行数: {len(ad_df)}")
    
    if len(ad_df) == 0:
        print("❌ 奇怪：原始数据里完全找不到 Alzheimer？")
        return

    # 2. 看看这些数据是什么类型的？(基因？表型？药物？)
    print(f"\n📊 {SEED_KEYWORD} 相关数据的 [Type] 分布 (Head <-> Tail):")
    # 创建一个临时列来看连接类型
    ad_df['conn_type'] = ad_df['x_type'] + " <-> " + ad_df['y_type']
    print(ad_df['conn_type'].value_counts().head(10)) # 只看前10种主要类型

    # 3. 看看具体的关系 (Relation)
    print(f"\n🔗 {SEED_KEYWORD} 相关数据的 [Relation] 分布:")
    print(ad_df['relation'].value_counts().head(10))

    # 4. 模拟“去药物化”后的结果
    # ADNI 研究通常关注病理机制（基因、蛋白、表型），而不是药物治疗
    print("\n🔍 模拟过滤药物 (Drug) 后的纯净生物学数据:")
    non_drug_ad = ad_df[
        (ad_df['x_type'] != 'drug') & 
        (ad_df['y_type'] != 'drug') & 
        (ad_df['relation'] != 'indication') & 
        (ad_df['relation'] != 'contraindication')
    ]
    
    print(f"✅ 发现 {len(non_drug_ad)} 条纯生物学关系 (基因/表型/通路)！")
    
    if len(non_drug_ad) > 0:
        print("   👀 预览前 10 条精华数据:")
        print(non_drug_ad[['x_name', 'relation', 'y_name', 'x_type', 'y_type']].head(10))
        
        # 5. 特别检查一下核心基因
        # 在 AD 中，APOE, APP, PSEN1, MAPT (Tau) 是核心，看看有没有
        print("\n🧬 核心 AD 基因检查 (APOE, APP, MAPT):")
        core_genes = ['APOE', 'APP', 'MAPT', 'PSEN1']
        for gene in core_genes:
            gene_hits = non_drug_ad[
                (non_drug_ad['x_name'] == gene) | (non_drug_ad['y_name'] == gene)
            ]
            print(f"   - {gene}: {len(gene_hits)} 条关联")

if __name__ == "__main__":
    inspect_data()

🕵️ 正在侦察 PrimeKG (kg.csv) 关于 'Alzheimer' 的数据 ...
   PrimeKG 原始总行数: 8100498
👉 包含 'Alzheimer' 的相关行数: 784

📊 Alzheimer 相关数据的 [Type] 分布 (Head <-> Tail):
conn_type
gene/protein <-> disease        205
disease <-> gene/protein        205
drug <-> disease                 73
disease <-> drug                 73
disease <-> effect/phenotype     71
effect/phenotype <-> disease     71
disease <-> disease              24
gene/protein <-> pathway         22
pathway <-> gene/protein         22
exposure <-> disease              8
Name: count, dtype: int64

🔗 Alzheimer 相关数据的 [Relation] 分布:
relation
disease_protein               410
disease_phenotype_positive    142
contraindication              116
pathway_protein                44
indication                     30
disease_disease                24
exposure_disease               16
pathway_pathway                 2
Name: count, dtype: int64

🔍 模拟过滤药物 (Drug) 后的纯净生物学数据:
✅ 发现 638 条纯生物学关系 (基因/表型/通路)！
   👀 预览前 10 条精华数据:
                                       